Main variables

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime

base_path = "s3://.../events/"
sf_target_table = "EVENTS_LANDING"

sfOptions = {
  "sfURL": dbutils.secrets.get("snowflake_retail_scope", "snowflake_host"),
  "sfUser": dbutils.secrets.get("snowflake_retail_scope", "snowflake_user"),
  "sfPassword": dbutils.secrets.get("snowflake_retail_scope", "snowflake_password"),
  "sfDatabase": dbutils.secrets.get("snowflake_retail_scope", "snowflake_database"),
  "sfSchema": dbutils.secrets.get("snowflake_retail_scope", "snowflake_schema"),
  "sfWarehouse": dbutils.secrets.get("snowflake_retail_scope", "snowflake_warehouse"),
  "sfRole": dbutils.secrets.get("snowflake_retail_scope", "snowflake_role"),
  "tempDir": "s3://.../sf_temp/"
}

Missing dates

In [ ]:
date_folders = dbutils.fs.ls(base_path) 
    # [dbutils] Databricks Utilities. 
    # [fs] File System (para trabajar con archivos o rutas). 
    # [ls] List (para listar lo que hay dentro).
    # Es decir, va al path de la carpeta y lista todo lo que hay adentro, es decir los "event_date=<fecha>/"

available_dates = []
for it in date_folders:
    name = it.name.rstrip("/")
    if name.startswith("event_date="):
        date_str = name.replace("event_date=", "")
        try:
            d = datetime.strptime(date_str, "%Y-%m-%d").date()
            available_dates.append(d)
        except:
            pass

available_dates_set= set(available_dates) # set() crea un conjunto, que luego es facil para ver que fechas faltan: set1 - set2

loaded_dates_df = (
    spark.read
         .format("snowflake")
         .options(**sfOptions)
         .option("query", f"SELECT DISTINCT EVENT_DATE FROM {sf_target_table}")
         .load()
)

loaded_dates_set = set()
for it in loaded_dates_df.collect(): # collect() en spark trae todos los registros de un df como una lista de filas
    d = it["EVENT_DATE"]
    loaded_dates_set.add(d.date() if hasattr(d, "date") else d) # Checkea que sea tipo date, sino si es datetime lo pasa.

missing_dates_set = available_dates_set - loaded_dates_set

Data transformation and write

In [ ]:
paths = [f"{base_path}event_date={d}" for d in missing_dates_set]

if len(paths) == 0:
    print("No hay fechas nuevas para cargar.")
else:
    df_raw = spark.read.parquet(*paths)

    df_clean = (
        df_raw
        # ------- Tiempo -------
        .withColumn("event_time", F.col("event_time").cast("timestamp"))
        .withColumn("event_date", F.to_date("event_time"))
        .withColumn("load_date", F.col("load_date").cast("timestamp"))
        # ------ Numeros -------
        .withColumn("price", F.col("price").cast("decimal(10,2)"))
        .withColumn("product_id", F.col("product_id").cast("long"))
        .withColumn("user_id", F.col("user_id").cast("long"))
        .withColumn("category_id", F.col("category_id").cast("long"))
        # ------ Strings -------
        .withColumn("event_type", F.trim(F.col("event_type")).cast("string"))
        .withColumn("category_code", F.trim(F.col("category_code")).cast("string"))
        .withColumn("brand", F.trim(F.col("brand")).cast("string"))
        .withColumn("user_session", F.trim(F.col("user_session")).cast("string"))
        # ------ Metadata ------ 
        .drop("__index_level_0__") # Corregir esto en el codigo para subir los docs a S3
    )

    cols_sf_order = [
        "event_time",
        "event_date",
        "load_date",
        "price",
        "product_id",
        "user_id",
        "category_id",
        "event_type",
        "category_code",
        "brand",
        "user_session"
        ]

    df_clean = df_clean.select(*cols_sf_order)

    (
        df_clean.write
            .format("snowflake")
            .options(**sfOptions)
            .option("dbtable", sf_target_table)
            .mode("append")
            .save()
    )

    print("Cargado a Snowflake:", len(paths), "carpetas")